<a href="https://colab.research.google.com/github/kumar23kan/conformerflow/blob/claude%2Fsetup-model-training-E3Cdr/colab_pro_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ConformerFlow — Colab Pro Training

**Before running:**
1. `Runtime → Change runtime type → A100 GPU`
2. Run cells top to bottom
3. Keep this tab open (or use Colab Pro background execution)

**What you need in Google Drive:**
- `MyDrive/conformerflow/ckpt_latest.pt` — checkpoint from Kaggle session2
- `MyDrive/conformerflow/conformerflow_data/` — parsed NMR data (optional, can re-download)

**How to get checkpoint from Kaggle:**
1. Go to `kaggle.com/datasets/perinbamkumar/conformerflow-ckpt-session2`
2. Click Download → extract → find `ckpt_latest.pt`
3. Upload to Google Drive at `MyDrive/conformerflow/ckpt_latest.pt`

In [1]:
# Cell 1 — Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('BF16 support:', torch.cuda.is_bf16_supported())

GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
BF16 support: True


In [2]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

Path('/content/drive/MyDrive/conformerflow').mkdir(parents=True, exist_ok=True)
print('Drive mounted. conformerflow folder ready.')
print('Contents:', os.listdir('/content/drive/MyDrive/conformerflow'))

Mounted at /content/drive
Drive mounted. conformerflow folder ready.
Contents: ['ckpt_latest.pt']


In [3]:
# Cell 3 — Clone repo and install
import shutil, os
from pathlib import Path

if Path('/content/conformerflow').exists():
    shutil.rmtree('/content/conformerflow')

!git clone -b claude/setup-model-training-E3Cdr \
    https://github.com/kumar23kan/conformerflow.git \
    /content/conformerflow

!pip install -q biopython gemmi requests tqdm numpy einops scipy
print('Done')

Cloning into '/content/conformerflow'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 270 (delta 8), reused 15 (delta 6), pack-reused 252 (from 1)
Receiving objects: 100% (270/270), 269.31 KiB | 26.93 MiB/s, done.
Resolving deltas: 100% (157/157), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 110.9 MB/s eta 0:00:00
Done


In [4]:
# Cell 4 — Set working directory
import os, sys
os.chdir('/content/conformerflow/files')
sys.path.insert(0, '/content/conformerflow/files')
print('Working directory:', os.getcwd())

Working directory: /content/conformerflow/files


In [5]:
# Cell 5 — Load parsed data
# Option A: Copy from Drive (if you already uploaded it)
# Option B: Download from Kaggle using API

import shutil, os, json
from pathlib import Path

for d in ['pdb_data/parsed_nmr', 'pdb_data/splits', 'checkpoints']:
    Path(d).mkdir(parents=True, exist_ok=True)

drive_data = '/content/drive/MyDrive/conformerflow/conformerflow_data'

if Path(drive_data).exists():
    print('Loading data from Drive...')
    shutil.copytree(f'{drive_data}/parsed_nmr', 'pdb_data/parsed_nmr', dirs_exist_ok=True)
    shutil.copytree(f'{drive_data}/splits',     'pdb_data/splits',     dirs_exist_ok=True)
    print('Data loaded from Drive')
else:
    print('Drive data not found. Downloading from Kaggle...')
    kaggle_json = Path('/content/kaggle.json')
    if kaggle_json.exists():
        Path('/root/.kaggle').mkdir(exist_ok=True)
        shutil.copy(kaggle_json, '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
        !kaggle datasets download -d perinbamkumar/conformerflow-data -p /content/kdata --unzip
        shutil.copytree('/content/kdata/parsed_nmr', 'pdb_data/parsed_nmr', dirs_exist_ok=True)
        shutil.copytree('/content/kdata/splits',     'pdb_data/splits',     dirs_exist_ok=True)
        print('Data downloaded from Kaggle')
    else:
        print('ERROR: Upload kaggle.json to /content/kaggle.json')

for split in ['train', 'val', 'test']:
    path = f'pdb_data/splits/{split}.json'
    if Path(path).exists():
        data = json.load(open(path))
        print(f'{split}: {len(data)} entries')

Drive data not found. Downloading from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/perinbamkumar/conformerflow-data
License(s): CC0-1.0
100% 703M/703M [00:38<00:00, 19.0MB/s]

Data downloaded from Kaggle
train: 6364 entries
val: 1044 entries
test: 1046 entries


In [6]:
# Cell 6 — Load checkpoint from Drive
import shutil, os
from pathlib import Path

ckpt_sources = [
    '/content/drive/MyDrive/conformerflow/ckpt_latest.pt',
    '/content/drive/MyDrive/conformerflow/ckpt_best.pt',
    '/content/drive/MyDrive/conformerflow/checkpoints/ckpt_latest.pt',
]

loaded = False
for src in ckpt_sources:
    if Path(src).exists():
        shutil.copy(src, 'checkpoints/ckpt_resume.pt')
        size = Path(src).stat().st_size / 1e6
        print(f'Checkpoint loaded: {src} ({size:.0f} MB)')
        loaded = True
        break

if not loaded:
    print('ERROR: No checkpoint found in Drive.')
    print('Upload ckpt_latest.pt to MyDrive/conformerflow/')

Checkpoint loaded: /content/drive/MyDrive/conformerflow/ckpt_latest.pt (161 MB)


In [ ]:
# Cell 7 — Train (A100 optimized: bf16, batch_size=32)
# Saves checkpoint to Drive every 2000 steps automatically
!cd /content/conformerflow/files && python3 train.py \
    --config configs/base_config.yaml \
    --batch_size 32 \
    --max_epochs 200 \
    --max_residues 300 \
    --bf16 \
    --resume_from checkpoints/ckpt_resume.pt

2026-05-19 10:55:56,110 [INFO] Config loaded from configs/base_config.yaml
2026-05-19 10:55:56,151 [INFO]   cfg  representation.structure: se3_frames
2026-05-19 10:55:56,151 [INFO]   cfg  representation.sequence: onehot
2026-05-19 10:55:56,151 [INFO]   cfg  model.d_model: 256
2026-05-19 10:55:56,151 [INFO]   cfg  model.d_latent: 16
2026-05-19 10:55:56,151 [INFO]   cfg  model.n_encoder_layers: 4
2026-05-19 10:55:56,151 [INFO]   cfg  model.n_flow_layers: 8
2026-05-19 10:55:56,151 [INFO]   cfg  model.n_heads: 8
2026-05-19 10:55:56,151 [INFO]   cfg  model.n_points: 4
2026-05-19 10:55:56,151 [INFO]   cfg  model.d_ff: 512
2026-05-19 10:55:56,151 [INFO]   cfg  model.dropout: 0.1
2026-05-19 10:55:56,151 [INFO]   cfg  generative_model.type: flow_matching
2026-05-19 10:55:56,151 [INFO]   cfg  generative_model.sigma_min: 0.01
2026-05-19 10:55:56,151 [INFO]   cfg  generative_model.ode_method: heun
2026-05-19 10:55:56,151 [INFO]   cfg  generative_model.n_inference_steps: 20
2026-05-19 10:55:56,151 

In [ ]:
# Cell 8 — Save checkpoints to Google Drive
import shutil, os
from pathlib import Path

src = Path('/content/conformerflow/files/checkpoints')
dst = Path('/content/drive/MyDrive/conformerflow/checkpoints')
dst.mkdir(parents=True, exist_ok=True)

if src.exists() and list(src.glob('*.pt')):
    for f in src.glob('*.pt'):
        shutil.copy(f, dst / f.name)
        size = f.stat().st_size / 1e6
        print(f'Saved: {f.name} ({size:.0f} MB)')
    print(f'All checkpoints saved to Drive: {dst}')
else:
    print('No checkpoints found.')